# Week 9: Transformers & LLM Architectures
## Notebook 08: Grouped-Query Attention (GQA) & LLaMA 3 Architecture

### 1. Conceptual Intuition & Theoretical Background
While Multi-Query Attention (MQA) dramatically cuts KV cache size, it can degrade model capacity on complex reasoning tasks. Multi-Head Attention (MHA) has maximal accuracy but massive memory bandwidth overhead.

**Grouped-Query Attention (GQA)** interpolates between MHA and MQA:
- Query heads ($H_Q$) are partitioned into $G$ groups.
- Each group shares a single Key and Value head ($H_{KV} = G$).
- Ratio:
$$\text{Group Size } S_G = \frac{H_Q}{H_{KV}}$$

$$\text{If } H_{KV} = H_Q \implies \text{MHA}$$
$$\text{If } H_{KV} = 1 \implies \text{MQA}$$
$$\text{If } 1 < H_{KV} < H_Q \implies \text{GQA (e.g. LLaMA-3 uses 32 query heads, 8 KV groups)}$$

---

### 2. Scratch Python Implementation: Modular GQA Layer

In [ ]:
import numpy as np

class GroupedQueryAttentionScratch:
    def __init__(self, num_q_heads=32, num_kv_heads=8, head_dim=64):
        self.num_q = num_q_heads
        self.num_kv = num_kv_heads
        self.head_dim = head_dim
        self.group_size = num_q_heads // num_kv_heads
        
    def forward(self, Q, K, V):
        """
        Q: (batch, num_q, 1, head_dim)
        K: (batch, num_kv, seq_len, head_dim)
        V: (batch, num_kv, seq_len, head_dim)
        """
        batch_size, _, _, _ = Q.shape
        seq_len = K.shape[2]
        
        # Expand/Repeat KV heads to match Q heads
        # Shape becomes (batch, num_q, seq_len, head_dim)
        K_expanded = np.repeat(K, repeats=self.group_size, axis=1)
        V_expanded = np.repeat(V, repeats=self.group_size, axis=1)
        
        # Scaled dot-product attention
        scores = np.matmul(Q, K_expanded.transpose(0, 1, 3, 2)) / np.sqrt(self.head_dim)
        attn_weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
        attn_weights /= np.sum(attn_weights, axis=-1, keepdims=True)
        
        out = np.matmul(attn_weights, V_expanded)
        return out

# Simulate LLaMA-3 Configuration
# 32 Query Heads grouped into 8 KV Groups (each group shared by 4 Query heads)
gqa = GroupedQueryAttentionScratch(num_q_heads=32, num_kv_heads=8, head_dim=64)

batch = 2
seq_len = 128
Q_step = np.random.randn(batch, 32, 1, 64)
K_cache = np.random.randn(batch, 8, seq_len, 64)
V_cache = np.random.randn(batch, 8, seq_len, 64)

out = gqa.forward(Q_step, K_cache, V_cache)
print(f"GQA Output Shape: {out.shape} (Matches Query heads: 32)")
print(f"KV Cache savings relative to standard MHA: {(1 - 8/32)*100:.1f}%")

### 3. Step-by-Step Code Walkthrough
1. **Head Ratio ($rac{H_Q}{H_{KV}}$)**: Groups 4 query heads to 1 key/value head.
2. **`np.repeat`**: Replicates KV tensors across group members during attention calculation.
3. **Pareto Optimal**: Recovers $99\%$ of MHA model quality while delivering near-MQA inference speeds.

---

### 4. Industry Applications & Trade-offs
- **Open-Weights Foundation Models:** LLaMA 2 (70B), LLaMA 3 (8B/70B), and Mistral 7B all standardize on 8-group GQA.
- **Trade-off:** Slightly higher memory footprint than pure MQA.

---

### 5. Top Interview Questions & Answers
**Q1: Why did Meta choose Grouped-Query Attention (GQA) for LLaMA-3 instead of pure MQA or standard MHA?**
> **Answer:** Standard MHA suffers from memory bottlenecks during long-context generation, while pure MQA causes performance drops on complex math/code benchmarks. GQA strikes the optimal balance between high perplexity quality and fast throughput.

**Q2: What is "Uptraining" in the context of converting MHA models to GQA?**
> **Answer:** Rather than training from scratch, an MHA checkpoint's Key and Value heads are mean-pooled into $G$ groups, followed by training for a small fraction ($approx 5\%$) of the original pretraining tokens to recover performance.